In [ ]:
# SVM implementation
import cvxpy as cvx
import numpy as np
import numpy as np
import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import Dataset, DataLoader
from linformer import Linformer 
from itertools import chain   
from vit_pytorch.efficient import ViT   
data = {}

#data['test']  = np.load('drive/MyDrive/dataverse_files/Dataset5_raw_test.npz')
data['train'] = dict(np.load('dataverse_files/Dataset5_raw_train.npz'))
data['val']   = dict(np.load('dataverse_files/Dataset5_raw_val.npz'))
#### CONSTANTS
FEATURE_SIZE = 1000

In [ ]:
class SVM:
  def __init__(self, feature_size):
    self.data = data
    self.feature_size = feature_size
    self.w = None
    self.b = None

  def fit(self, X, label, C=10):
    self.w = cvx.Variable(FEATURE_SIZE)
    self.b = cvx.Variable(1)
    slack_vars = cvx.Variable(len(label))

    objective = cvx.Minimize((cvx.square(cvx.norm(self.w))/2) + (C * cvx.norm(slack_vars, 1)))
    print(self.w.shape)
    
    constraints = [slack_vars >= 0]
    
    for i in range(len(label)):
      constraints.append(label[i]*(X[i,:] @ self.w + self.b) + slack_vars[i] - 1 >= 0)
    
    optim_prob = cvx.Problem(objective, constraints)
    sol = optim_prob.solve(solver='CVXOPT', verbose=True)


    if optim_prob.status not in ["infeasible", "unbounded"]:
      print("The optimization for SVM was successfully solved.")
    else:
      print("Unable to solve the optimization problem")

  def predict(self, x):
    return self.w.T @ x + self.b

In [ ]:
# imports
import numpy as np

data = {}
#data['test']  = np.load('drive/MyDrive/dataverse_files/Dataset5_raw_test.npz')
data['train'] = dict(np.load('dataverse_files/Dataset5_raw_train.npz'))
data['val']   = dict(np.load('dataverse_files/Dataset5_raw_val.npz'))

In [ ]:
#feature extraction from the image
## importing the necessary modules for feature extraction 
import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import Dataset, DataLoader
import pickle
# normalize image data
# N x 224 x 224 x3 
# take avg of N images 

mean = data['train']['image'].mean(axis=(0,1,2), keepdims=True)
devia = data['train']['image'].std(axis=(0,1,2), keepdims=True)
data['train']['image'] = (data['train']['image'] - mean) / devia
data['val']['image'] = (data['val']['image'] - mean) / devia

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, data, labels, transform):
        self.transform = transform
        self.data = data
        self.labels = labels
    
    def __len__(self):
        return self.labels.size

    def __getitem__(self, idx):
        temp = int(self.labels[idx])
            
        return self.transform(self.data[idx]).float(), temp

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])
print(data['train']['image'].transpose(0, 3, 1, 2).shape)
trainset = CustomDataset(data['train']['image'], data['train']['image_label'], transform)
train_loader = DataLoader(trainset, 64, shuffle=True, drop_last=True)